# Weighted ensemble of ConvNeXt, EfficientNetV2-S, BiT and HRNet

The equal-weight mean of Exp1, Exp3 and Exp5 scored 0.98047 on the public leaderboard. Adding HRNet (Exp7) with equal weight lowered it to 0.97816 (notebook 18). Eight named-pet cells explain the difference, and four of the five that were removed had been called by Exp1 alone, which cannot pass the 0.25 threshold in an equal mean of four. This notebook asks whether giving the members different weights keeps what HRNet adds without losing those cells.

**What we can and cannot learn.** Fold 0 has 8–10 photos of each named pet, three of the four members were selected on it, and every weighting we try is within about ±0.0005 macro F1 of the equal mean there. Fold 0 therefore cannot choose weights, and fitting them on it would fit noise (the stacking in notebook 15 did, and lost 0.0136 on the leaderboard). So we do not search. We define a few weightings from simple rules, show exactly which test cells each one changes, and let the leaderboard judge them. Each upload also reads the public part of the test set, so a gain of a few thousandths may not carry to the private score.

By default the notebook only reads saved probabilities. Submission CSVs and run records are written only with `BUILD_SUBMISSION=True`, and an existing file is kept, not overwritten. Kaggle upload is always manual.

In [1]:
from __future__ import annotations

import json
import subprocess
import sys
import os
from pathlib import Path

import numpy as np

BUILD_SUBMISSION = True  # Writes one CSV and one run record per weighting; existing files are kept.
SUBMIT_TO_KAGGLE = False  # No submission API call exists in this notebook.

VALIDATION_FOLD = 0
THRESHOLD = 0.25
# Experiment folder -> completed run ID. All four have saved fold-0 and test probabilities.
MEMBERS = {
    'exp1': 'convnext_standard_fold0_seed42_20261018_full1',
    'exp3': 'efficientnetv2s_fold0_seed42',
    'exp5': 'bit_r50x1_fold0_seed42',
    'exp07': 'hrnet_w32_finetune_fold0_seed42',
}
# Public leaderboard scores of the single models and the competition's solid baseline.
PUBLIC_SCORE = {'exp1': 0.97003, 'exp3': 0.97015, 'exp5': 0.94035, 'exp07': 0.96431}
SOLID_BASELINE = 0.90624

# Weightings from simple rules (no search). The equal means are the references.
WEIGHTS = {
    'eq3 (public 0.98047)': {'exp1': 1, 'exp3': 1, 'exp5': 1, 'exp07': 0},
    'eq4 (public 0.97816)': {'exp1': 1, 'exp3': 1, 'exp5': 1, 'exp07': 1},
    # HRNet counts half: the smallest step away from the best known ensemble.
    'hrnet_half': {'exp1': 1, 'exp3': 1, 'exp5': 1, 'exp07': 0.5},
    # Each member weighted by how far its public score is above the solid baseline.
    'skill': {name: PUBLIC_SCORE[name] - SOLID_BASELINE for name in MEMBERS},
    # The two strongest single models count double.
    'top2_double': {'exp1': 2, 'exp3': 2, 'exp5': 1, 'exp07': 1},
}
REFERENCE = 'eq3 (public 0.98047)'
TO_SUBMIT = ('hrnet_half', 'skill', 'top2_double')  # The files to write; the equal means already exist.
BOOTSTRAPS = 300
print('weightings (Exp1, Exp3, Exp5, Exp7):')
for title, weights in WEIGHTS.items():
    print(f'  {title:<24}', [round(float(w), 3) for w in np.asarray(list(weights.values()), float) / sum(weights.values())])

weightings (Exp1, Exp3, Exp5, Exp7):
  eq3 (public 0.98047)    [0.333, 0.333, 0.333, 0.0]
  eq4 (public 0.97816)    [0.25, 0.25, 0.25, 0.25]
  hrnet_half              [0.286, 0.286, 0.286, 0.143]
  skill                   [0.29, 0.291, 0.155, 0.264]
  top2_double             [0.333, 0.333, 0.167, 0.167]


## Load the members' predictions

As in notebooks 13 and 18, each run's fold-0 probabilities are re-scored and must match its run record, the members must share the fold file, label order and threshold, and a member's saved test probabilities are used only if they come from the recorded checkpoint. No model is run.

In [2]:
start = Path(os.environ.get('STAGE2_ROOT', Path.cwd())).resolve()
ROOT = next((path for path in (start, *start.parents) if (path / 'folds.csv').is_file() and (path / 'stage2' / 'experiment.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Launch Jupyter from this repository or a subdirectory.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from stage2.ensemble import read_probabilities, weighted_mean_probabilities
from stage2.experiment import LABEL_COLUMNS, form_submission, read_csv, score_probabilities, split_rows, write_probabilities, write_run_record, write_submission
from stage2.run_records import file_sha256

DATA_DIR = ROOT / 'data'
_, validation_rows = split_rows(DATA_DIR, ROOT / 'folds.csv', VALIDATION_FOLD)
validation_ids = [row['image_id'] for row in validation_rows]
validation_targets = np.asarray([[int(row[label]) for label in LABEL_COLUMNS] for row in validation_rows])
test_ids = [row['image_id'] for row in read_csv(DATA_DIR / 'test.csv', ('image_id',))]
fold_sha256 = file_sha256(ROOT / 'folds.csv')

records, validation_probabilities, test_probabilities = {}, {}, {}
for name, run_id in MEMBERS.items():
    run_dir = ROOT / 'experiments' / name / 'results' / run_id
    record = json.loads((run_dir / f'{run_id}.json').read_text())
    if record['status'] != 'completed' or record['fold'] != VALIDATION_FOLD:
        raise ValueError(f'{run_id} is not a completed fold-{VALIDATION_FOLD} run.')
    if record['data']['fold_sha256'] != fold_sha256 or record['data']['label_columns'] != list(LABEL_COLUMNS):
        raise ValueError(f'{run_id} used a different fold file or label order.')
    if record['metrics']['threshold'] != THRESHOLD:
        raise ValueError(f'{run_id} was scored at another threshold.')
    probabilities = read_probabilities(run_dir / f'{run_id}_val.csv', validation_ids)
    rescored = score_probabilities(validation_targets, probabilities, THRESHOLD)['macro_f1']
    if not np.isclose(rescored, record['metrics']['macro_f1'], rtol=0, atol=1e-9):
        raise ValueError(f'{run_id}: re-scored macro F1 {rescored} differs from its record.')
    test_dir = ROOT / 'experiments' / name / 'results' / f'{run_id}_test'
    cached = json.loads((test_dir / f'{run_id}_test.json').read_text())
    if cached['source_checkpoint_sha256'] != record['artifacts']['best_checkpoint_sha256']:
        raise ValueError(f'{run_id}: saved test probabilities come from another checkpoint.')
    records[name] = record
    validation_probabilities[name] = probabilities
    test_probabilities[name] = read_probabilities(test_dir / f'{run_id}_test.csv', test_ids)
    print(f'{name}: {run_id} | fold-0 macro F1 {rescored:.5f} (matches record) | public {PUBLIC_SCORE[name]}')

exp1: convnext_standard_fold0_seed42_20261018_full1 | fold-0 macro F1 0.97518 (matches record) | public 0.97003
exp3: efficientnetv2s_fold0_seed42 | fold-0 macro F1 0.99831 (matches record) | public 0.97015
exp5: bit_r50x1_fold0_seed42 | fold-0 macro F1 0.99851 (matches record) | public 0.94035
exp07: hrnet_w32_finetune_fold0_seed42 | fold-0 macro F1 0.99733 (matches record) | public 0.96431


## Fold 0: it cannot tell the weightings apart

Macro F1 at 0.25 and macro AP for every weighting, and the macro-F1 difference from the reference mean over 300 bootstrap resamples of the 2,452 fold-0 images (5th to 95th percentile). An interval that contains zero means fold 0 does not separate the two. This is the reason we do not tune the weights here.

In [3]:
member_names = list(MEMBERS)


def combine(probabilities_by_member: dict, weights: dict) -> np.ndarray:
    active = [name for name in member_names if weights[name] > 0]
    return weighted_mean_probabilities([probabilities_by_member[name] for name in active], [weights[name] for name in active])


validation_ensembles = {title: combine(validation_probabilities, weights) for title, weights in WEIGHTS.items()}
generator = np.random.default_rng(0)
resamples = [generator.integers(0, len(validation_targets), len(validation_targets)) for _ in range(BOOTSTRAPS)]
bootstrap_f1 = {title: np.array([score_probabilities(validation_targets[rows], probabilities[rows], THRESHOLD)['macro_f1'] for rows in resamples])
                for title, probabilities in validation_ensembles.items()}
print(f"{'weighting':<24} {'normalized weights':<26} {'F1@0.25':>8} {'AP':>8}  {'dF1 vs reference [5%, 95%]':>28}")
for title, probabilities in validation_ensembles.items():
    scores = score_probabilities(validation_targets, probabilities, THRESHOLD)
    weights = np.asarray([WEIGHTS[title][name] for name in member_names], float)
    difference = bootstrap_f1[title] - bootstrap_f1[REFERENCE]
    print(f"{title:<24} {str(np.round(weights / weights.sum(), 2).tolist()):<26} {scores['macro_f1']:>8.5f} {scores['macro_ap']:>8.5f}  "
          f"{difference.mean():>+9.4f} [{np.percentile(difference, 5):+.4f}, {np.percentile(difference, 95):+.4f}]")

weighting                normalized weights          F1@0.25       AP    dF1 vs reference [5%, 95%]
eq3 (public 0.98047)     [0.33, 0.33, 0.33, 0.0]     0.98656  0.99999    +0.0000 [+0.0000, +0.0000]
eq4 (public 0.97816)     [0.25, 0.25, 0.25, 0.25]    0.99832  0.99999    +0.0115 [-0.0002, +0.0344]
hrnet_half               [0.29, 0.29, 0.29, 0.14]    0.98656  0.99999    -0.0000 [-0.0002, +0.0002]
skill                    [0.29, 0.29, 0.16, 0.26]    0.98646  0.99999    -0.0001 [-0.0005, +0.0003]
top2_double              [0.33, 0.33, 0.17, 0.17]    0.98666  0.99999    +0.0001 [-0.0002, +0.0004]


## The test set: which cells does each weighting change?

Against the reference mean (Exp1 + Exp3 + Exp5, public 0.98047) we count the cells each weighting changes. A named-pet cell is worth about 0.0007 macro F1 on the test set and a Cat or Dog cell about 0.00001, so only the named-pet columns can move the score noticeably. Then every named-pet cell that a candidate changes is listed with each member's probability, so the reason for each change can be read directly. The labels are hidden, so this shows who voted for a change, not whether it is right.

In [4]:
test_ensembles = {title: combine(test_probabilities, weights) for title, weights in WEIGHTS.items()}
reference = test_ensembles[REFERENCE] >= THRESHOLD
named_pets = [LABEL_COLUMNS.index(label) for label in ('Moris', 'Motya', 'Biatrix')]
print(f"{'weighting':<24} " + ' '.join(f'{label:>8}' for label in LABEL_COLUMNS) + f" {'cells vs ref':>13} {'named':>6}")
for title, probabilities in test_ensembles.items():
    decided = probabilities >= THRESHOLD
    changed = decided != reference
    print(f'{title:<24} ' + ' '.join(f'{int(decided[:, index].sum()):>8}' for index in range(len(LABEL_COLUMNS))) + f' {int(changed.sum()):>13} {int(changed[:, named_pets].sum()):>6}')

for title in TO_SUBMIT:
    decided = test_ensembles[title] >= THRESHOLD
    print(f'\n{title}: named-pet cells that differ from the reference')
    print(f"  {'label':<8} {'change':<8} " + ' '.join(f'{name:>6}' for name in member_names) + f" {'ref':>6} {'this':>6}")
    for label_index in named_pets:
        for row in np.flatnonzero(decided[:, label_index] != reference[:, label_index]):
            values = ' '.join(f'{test_probabilities[name][row, label_index]:>6.3f}' for name in member_names)
            print(f"  {LABEL_COLUMNS[label_index]:<8} {'added' if decided[row, label_index] else 'removed':<8} {values} "
                  f"{test_ensembles[REFERENCE][row, label_index]:>6.3f} {test_ensembles[title][row, label_index]:>6.3f}")

weighting                     Cat      Dog    Moris    Motya  Biatrix  cells vs ref  named
eq3 (public 0.98047)         7833     7718      167      137      185             0      0
eq4 (public 0.97816)         7825     7700      165      138      184            58      8
hrnet_half                   7827     7714      168      137      185            17      3
skill                        7826     7701      170      135      185            65     11
top2_double                  7820     7701      167      135      185            48      8

hrnet_half: named-pet cells that differ from the reference
  label    change     exp1   exp3   exp5  exp07    ref   this
  Moris    added     0.574  0.071  0.006  0.488  0.217  0.256
  Motya    added     0.310  0.057  0.184  0.822  0.184  0.275
  Motya    removed   0.277  0.028  0.476  0.001  0.260  0.223

skill: named-pet cells that differ from the reference
  label    change     exp1   exp3   exp5  exp07    ref   this
  Moris    added     0.574  0

## Write the submission files

One CSV per weighting in `TO_SUBMIT` (threshold 0.25, sample-submission order), plus the weighting's fold-0 probabilities and a run record under `experiments/exp4/results/<run-id>/`. Existing files are kept. The equal-weight means were written by notebooks 13 and 18. Upload by hand, and record the public score in the run record and the README.

In [5]:
git_commit = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=ROOT, capture_output=True, text=True, check=True).stdout.strip() if (ROOT / '.git').exists() else None
source_files = (ROOT / 'notebooks' / '19_weighted_ensemble_exp1_exp3_exp5_exp7.ipynb', ROOT / 'stage2' / 'ensemble.py', ROOT / 'stage2' / 'experiment.py')
source_sha256 = {str(path.relative_to(ROOT)): file_sha256(path) for path in source_files if path.is_file()}

if not BUILD_SUBMISSION:
    print('No submission CSV built or uploaded.')
for title in TO_SUBMIT if BUILD_SUBMISSION else ():
    run_id = f'weighted_{title}_exp1_exp3_exp5_exp7_fold0'
    result_dir = ROOT / 'experiments' / 'exp4' / 'results' / run_id
    submission_path = ROOT / 'submissions' / f"exp4_{run_id.removesuffix('_fold0')}_025.csv"
    targets = (submission_path, result_dir / f'{run_id}.json', result_dir / f'{run_id}_val.csv')
    if any(path.exists() for path in targets):
        print(f'{title}: already written, kept:', submission_path.relative_to(ROOT))
        continue
    weights = np.asarray([WEIGHTS[title][name] for name in member_names], float)
    scores = score_probabilities(validation_targets, validation_ensembles[title], THRESHOLD)
    write_submission(submission_path, form_submission(DATA_DIR / 'sample_submission.csv', test_ids, test_ensembles[title], THRESHOLD))
    write_probabilities(result_dir / f'{run_id}_val.csv', validation_ids, validation_ensembles[title], VALIDATION_FOLD)
    write_run_record(result_dir / f'{run_id}.json', {
        'run_id': run_id, 'experiment': '04_ensemble', 'status': 'completed', 'fold': VALIDATION_FOLD,
        'method': 'weighted mean of member probabilities (weights from a simple rule, not tuned), then fixed threshold',
        'weighting': title, 'weights': dict(zip(member_names, map(float, weights / weights.sum()))),
        'members': [{'experiment': records[name]['experiment'], 'run_id': MEMBERS[name],
                     'checkpoint_sha256': records[name]['artifacts']['best_checkpoint_sha256']} for name in member_names],
        'data': {'fold_file': 'folds.csv', 'fold_sha256': fold_sha256, 'label_columns': list(LABEL_COLUMNS)},
        'metrics': scores,
        'test_cells_changed_vs_reference': int((test_ensembles[title] >= THRESHOLD).__ne__(reference).sum()),
        'artifacts': {'validation_probabilities_csv': str((result_dir / f'{run_id}_val.csv').relative_to(ROOT)),
                      'submission_csv': str(submission_path.relative_to(ROOT))},
        'git_commit': git_commit, 'source_sha256': source_sha256,
        'public_leaderboard_score': None, 'error': None,
    })
    print(f'{title}: wrote', submission_path.relative_to(ROOT))

hrnet_half: wrote submissions/exp4_weighted_hrnet_half_exp1_exp3_exp5_exp7_025.csv
skill: wrote submissions/exp4_weighted_skill_exp1_exp3_exp5_exp7_025.csv


top2_double: wrote submissions/exp4_weighted_top2_double_exp1_exp3_exp5_exp7_025.csv


## Findings before submission

| Weighting | Weights (Exp1, Exp3, Exp5, Exp7) | Fold-0 F1 | Test cells vs 0.98047 | of them named-pet | Public leaderboard |
|---|---|---:|---:|---:|---:|
| eq3 (notebook 13) | .33 .33 .33 0 | 0.98656 | 0 | 0 | **0.98047** |
| eq4 (notebook 18) | .25 .25 .25 .25 | 0.99832 | 58 | 8 | 0.97816 |
| hrnet_half | .29 .29 .29 .14 | 0.98656 | 17 | 3 | to be filled |
| skill | .29 .29 .16 .26 | 0.98646 | 65 | 11 | to be filled |
| top2_double | .33 .33 .17 .17 | 0.98666 | 48 | 8 | to be filled |

**1. Fold 0 cannot choose between them.** The three weightings are within 0.0005 macro F1 of the reference, and every bootstrap interval contains zero. The equal four-model mean looks better (+0.0115), which is one Biatrix photo, and it lost 0.0023 on the leaderboard. Any weights chosen on fold 0 would fit noise, so these are fixed by simple rules and not searched.

**2. Each weighting changes only a handful of cells that matter.** Against the 0.98047 mean, `hrnet_half` changes 3 named-pet cells (adds a Moris that Exp1 and HRNet both half-support, adds a Motya that only HRNet is sure of at 0.82, removes a Motya that Exp5 0.48 and Exp1 0.28 half-support). `skill` changes 11 and `top2_double` 8. At about 0.0007 per named-pet cell, `hrnet_half` can move the score by at most about ±0.002, the other two by about ±0.008.

**3. `skill` and `top2_double` lower BiT's weight, and every named-pet cell they remove was BiT's.** They remove 5 cells where Exp5 gave 0.48 to 0.99 and the other members were low (for example Biatrix: Exp5 0.992, the rest 0.04 or less; Motya: Exp5 0.940 and 0.904). Notebook 13 found that BiT's votes were what lifted the three-model mean from 0.97761 to 0.98047, because every named-pet cell BiT added was a cell where Exp1 and Exp3 disagreed. The stacking submission, which also dropped single-member calls, lost 0.0136. We therefore expect `skill` and `top2_double` to score at or below 0.98047. They were written so that this expectation can be checked, but they are not the candidates to upload first.

**4. `hrnet_half` is the one consistent with the evidence.** It keeps BiT's and Exp1's weight, adds HRNet at half strength, and moves the score least. We do not expect a large gain. HRNet changed the three-model mean in 8 named-pet cells and the net effect was negative, so at half weight the realistic range is a few cells either way. If it beats 0.98047 by less than about 0.002, that is within what the public part of the test set can produce by chance and we would not read it as a better method.

**5. What would help more.** The equal-mean results (0.98047 with three members, 0.97816 with four) show that the voting rule at threshold 0.25 matters more than the weights: it changes with the number of members. Choosing the rule needs labelled out-of-fold predictions for all five folds (about 125 named-pet photos instead of 27), which we do not have for any member.

Record each public score in its run record under `experiments/exp4/results/weighted_*/` after the manual upload.